### Steps Involved

- Load environment variables from .env
- Read Groq and LangChain API keys
- Create Groq LLM (ChatGroq)
- Enable LangChain tracing (LangSmith)
- Import LangGraph components
- Define shared state (State) with messages
- Enable message accumulation using add_messages
- Create a StateGraph using the state
- Define chatbot node (calls the LLM)
- Add chatbot node to the graph
- Connect START → chatbot → END
- Compile the graph
- (Optional) Visualize the graph
- Start infinite user input loop
- Take user input from terminal
- Exit loop if user types q or quit
- Send user message into the graph
- Run the graph using stream()
- Receive node output events
- Extract assistant message
- Print assistant response
- Repeat until exit

In [ ]:
!pip install langgraph

In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
load_dotenv() 

groq_api_key=os.getenv("GROQ_API_KEY")
llm=ChatGroq(groq_api_key=groq_api_key,model='llama-3.1-8b-instant')

In [ ]:
os.environ['LANGCHAIN_API_KEY']=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_PROJECT"]="CourseLanggraph"

## Start Building Chatbot Using Langgraph

In [ ]:
from typing import Annotated # LangGraph uses Annotated to understand how state should behave 
from typing_extensions import TypedDict #  lets you define a dictionary with fixed keys and fixed value types.
from langgraph.graph import StateGraph,START,END # A flowchart that passes a shared state between steps
from langgraph.graph.message import add_messages # To handle chat messages in state  

In [ ]:
class State(TypedDict): # This defines the shape of the shared state that flows through your graph. 
    # Every node in this graph will receive and return a dictionary that looks like this. 
    messages:Annotated[list,add_messages] # List to store chat messages 

# Typically this list holds: Human messages ,AI messages,Tool messages
# Whenever a node returns new messages, append them to the existing list instead of replacing it. 

graph_builder=StateGraph(State)  

In [ ]:
def chatbot(state:State):
  return {"messages":llm.invoke(state['messages'])}

In [ ]:
graph_builder.add_node("chatbot",chatbot)

In [ ]:
graph_builder.add_edge(START,"chatbot")
graph_builder.add_edge("chatbot",END)
graph=graph_builder.compile()

In [ ]:
from IPython.display import Image, display
try:
  display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
  pass

In [ ]:
while True:
    user_input=input("User: ") 
    if user_input.lower() in [ 'q','quit']:  
        print("Bye")
        break 
    for event in  graph.stream({ "messages": [{"role": "user", "content": user_input}]}): 
        print(event.values()) 
        for value in event.values():
            print(value['messages'])
            print("Assistant:",value["messages"].content)